# ASD-XMF X3 Cohort Pipeline
Cohort audit and splits, face and audio pipelines, frame embeddings and all single-modality baselines (NB00-NB07).

Each section is one former notebook, run as its own process. A section waits for the ones it needs; a time guard stops before Kaggle's 12-hour limit and the next run continues where this one stopped.

| section | source | needs |
|---|---|---|
| `nb00` | `nb00_data_audit_manifest.py` | - |
| `nb01` | `nb01_face_pipeline.py` | nb00 |
| `nb02` | `nb02_audio_pipeline.py` | nb00 |
| `nb05` | `nb05_frame_embeddings.py` | nb01 |
| `nb07` | `nb07_audio_baselines.py` | nb02 |
| `nb06` | `nb06_video_baselines.py` | nb05 |
| `nb04a` | `nb04_frame_baselines.py` | nb01 |
| `nb04b` | `nb04_frame_baselines.py` | nb01 |
| `nb04c` | `nb04_frame_baselines.py` | nb01 |
| `nb04d` | `nb04_frame_baselines.py` | nb01 |

In [ ]:
RUN_PROFILE = "full"   # "full" on GPU; "preflight" = quick CPU smoke test (tools/push.py)
PREFLIGHT = RUN_PROFILE == "preflight"
import os, sys, glob, json, time, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

def _find_lib():
    for c in ("/kaggle/input/asd-xmf-lib", "/kaggle/input/datasets/shazedaoni/asd-xmf-lib"):
        if os.path.exists(os.path.join(c, "asd_utils.py")):
            return c
    for depth in range(1, 6):
        hits = glob.glob("/kaggle/input/" + "*/" * depth + "asd_utils.py")
        if hits:
            return os.path.dirname(hits[0])
    return os.path.abspath(os.path.join(os.getcwd(), "..", "..", "asd_lib"))  # local run

def _require_internet():
    import socket
    try:
        socket.create_connection(("pypi.org", 443), timeout=10).close()
    except OSError:
        raise SystemExit("NO INTERNET in this Kaggle session: verify the account's phone number (Kaggle > Settings) "
                         "and keep 'Internet on'. Stopping so no step runs with missing models or features.")
_require_internet()

LIB = _find_lib()
sys.path.insert(0, LIB)
import asd_utils as U
print("asd_lib:", LIB, "| profile:", RUN_PROFILE)
ENV = U.env_report()
open(os.path.join(str(U.WORK), "_run_profile_" + RUN_PROFILE + ".txt"), "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))  # queue marker

In [ ]:
import asd_runner as R
R.setup('shazedaoni/asd-xmf-x3-cohort-pipeline', RUN_PROFILE, [{"name": "nb00", "after": []}, {"name": "nb01", "after": ["nb00"]}, {"name": "nb02", "after": ["nb00"]}, {"name": "nb05", "after": ["nb01"]}, {"name": "nb07", "after": ["nb02"]}, {"name": "nb06", "after": ["nb05"]}, {"name": "nb04a", "after": ["nb01"]}, {"name": "nb04b", "after": ["nb01"]}, {"name": "nb04c", "after": ["nb01"]}, {"name": "nb04d", "after": ["nb01"]}], hours=11.0)

## Section `nb00`: NB00 - Cohort audit, duplicate check and locked child-wise splits

In [ ]:
%%writefile {R.SEC_DIR}/nb00.py
# # NB00 - Cohort audit, duplicate check and locked child-wise splits
# **Input.** The BanglaASD-XMF cohort as a private Kaggle dataset (attach it under
# *Add Input*) with this layout - nothing else is required:
# ```
# metadata/children.csv  child_id,label(1=ASD),group(ASD/TD/DD),age_months,sex,site,device,room,recorder,
#                        mchat_total,mchat_01..mchat_20
# metadata/clips.csv     child_id,clip_id,stimulus,video_path[,audio_path]   (paths relative to the dataset root)
# metadata/cry.csv       child_id,cry_id,cry_path                            (natural cries only - never provoked)
# ```
# If consent does not allow raw video in the cloud, run NB01/NB02 locally and upload only
# their feature outputs; every later notebook reads features, not video.
#
# **No cohort attached -> DEV mode.** A small *synthetic* cohort is generated from RAF-DB
# faces and synthetic tones, purely so NB01-NB12 can be executed and debugged before
# ethics approval.  Every DEV output is stamped SYNTHETIC; none of it is a result.
#
# **Output.** `manifest.csv` (one row per child + locked `split` + `fold_s0..2`),
# `clips.csv` with `clip_idx`, `cry.csv`, the audit report and the pHash duplicate report.
# 

import numpy as np, pandas as pd
from pathlib import Path
import cv2
OUT = U.WORK / "nb00"; OUT.mkdir(parents=True, exist_ok=True)
EXTERNAL_SITE = None      # e.g. "site2" -> that whole site becomes the locked test set (preferred)
TEST_FRAC = 0.20          # used when no external site is given
SEEDS = (0, 1, 2)
N_SPLITS = 5

root, is_dev = U.locate_cohort()
if root is None:
    raf = U.find_path("DATASET", want="dir")
    happy = sorted((raf / "train" / "4").glob("*.jpg"))
    neutral = sorted((raf / "train" / "7").glob("*.jpg"))
    spec = U.DevCohortSpec(n_children=16 if PREFLIGHT else 60, clips_per_child=3 if PREFLIGHT else 6)
    root = U.make_dev_cohort(OUT / "dev_cohort", happy, neutral, spec)
    is_dev = True
print("cohort root:", root, "| DEV:", is_dev)
if is_dev:
    print(U.SYNTHETIC_BANNER)


# ## 1. Schema checks
# 

children = pd.read_csv(root / "metadata" / "children.csv")
clips = pd.read_csv(root / "metadata" / "clips.csv")
cry = pd.read_csv(root / "metadata" / "cry.csv") if (root / "metadata" / "cry.csv").exists() else pd.DataFrame(columns=["child_id", "cry_id", "cry_path"])
problems = []
need = {"child_id", "label", "age_months", "sex"}
miss = need - set(children.columns)
if miss: problems.append(f"children.csv missing columns {miss}")
if children.child_id.duplicated().any(): problems.append("duplicate child_id in children.csv")
if not set(children.label.unique()) <= {0, 1}: problems.append("label must be 0/1 (1 = ASD)")
orph = set(clips.child_id) - set(children.child_id)
if orph: problems.append(f"{len(orph)} clip rows reference unknown children")
clips["exists"] = [(root / p).exists() for p in clips.video_path]
if not clips.exists.all(): problems.append(f"{(~clips.exists).sum()} video files missing")
if len(cry):
    cry["exists"] = [(root / p).exists() for p in cry.cry_path]
for c in ("site", "device", "room", "recorder", "group"):
    if c not in children:
        children[c] = "unknown"
print("schema problems:", problems or "none")
clips["clip_idx"] = clips.groupby("child_id").cumcount()   # shared clip order for NB01 and NB02


# ## 2. Cohort description and confound audit
# Device / room / recorder / site must be **independent of the label** - otherwise a model
# can learn the room instead of the child (plan 5.4).  Age and sex imbalance is reported
# and later handled in fairness and matched analyses.
# 

from scipy import stats
def cramers_v(tab):
    chi2 = stats.chi2_contingency(tab)[0]
    n = tab.values.sum(); r, k = tab.shape
    return float(np.sqrt(chi2 / (n * max(1, min(r, k) - 1))))
desc = children.groupby("label").agg(n=("child_id", "size"), age_mean=("age_months", "mean"), age_sd=("age_months", "std"),
                                     male=("sex", lambda s: (s == "M").mean()))
print(desc)
audit = []
for col in ("sex", "site", "device", "room", "recorder", "group"):
    tab = pd.crosstab(children[col], children.label)
    if tab.shape[0] > 1:
        p = stats.chi2_contingency(tab)[1]
        audit.append({"variable": col, "test": "chi2", "p": p, "cramers_v": cramers_v(tab),
                      "flag": "CONFOUND RISK" if (col in ("site", "device", "room", "recorder") and p < 0.05) else ""})
t = stats.mannwhitneyu(children.age_months[children.label == 1], children.age_months[children.label == 0])
audit.append({"variable": "age_months", "test": "mann-whitney", "p": t.pvalue, "cramers_v": np.nan, "flag": ""})
audit = pd.DataFrame(audit)
print(U.table_md(audit))
clip_per_child = clips.groupby("child_id").size()
print("clips per child:", clip_per_child.describe().round(1).to_dict())
print("children with cry:", cry.child_id.nunique() if len(cry) else 0)


# ## 3. Perceptual-hash duplicate check (before any split)
# Middle frame of every clip.  Near-identical frames from **different children** mean a
# file was copied or mislabelled and must be fixed at the source before training.
# 

def middle_frame(path):
    cap = cv2.VideoCapture(str(path)); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 1)
    cap.set(cv2.CAP_PROP_POS_FRAMES, n // 2); ok, fr = cap.read(); cap.release()
    return cv2.cvtColor(fr, cv2.COLOR_BGR2RGB) if ok else None
hashes, keys, owners = [], [], {}
for r in clips[clips.exists].itertuples():
    fr = middle_frame(root / r.video_path)
    if fr is not None:
        hashes.append(U.phash(fr)); keys.append(r.clip_id); owners[r.clip_id] = r.child_id
dups = U.near_duplicates(hashes, keys, max_dist=4)
if len(dups):
    dups["child_a"] = dups.a.map(owners); dups["child_b"] = dups.b.map(owners)
    cross = dups[dups.child_a != dups.child_b]
else:
    cross = dups
print(f"near-duplicate clip pairs: {len(dups)} | across different children: {len(cross)}")
cross.to_csv(OUT / "phash_cross_child_duplicates.csv", index=False)
if len(cross) and not is_dev:
    problems.append(f"{len(cross)} cross-child duplicate clips - resolve before training")


# ## 4. Locked splits (by child) - created once, read by every notebook
# 

man = U.make_child_splits(children, n_splits=N_SPLITS, seeds=SEEDS, test_frac=TEST_FRAC,
                          external_site=EXTERNAL_SITE, site_col="site")
for s in SEEDS:
    for f in range(N_SPLITS):
        tr, va, te = U.fold_ids(man, s, f)
        U.assert_no_leak(tr, va, te)
man["age_band"] = U.age_band(man.age_months).astype(str)
man["is_synthetic"] = is_dev
man["cohort_root"] = str(root)
print(man.groupby(["split", "label"]).size().unstack())
man.to_csv(OUT / "manifest.csv", index=False)
clips.to_csv(OUT / "clips.csv", index=False)
cry.to_csv(OUT / "cry.csv", index=False)
report = {"is_synthetic": bool(is_dev), "n_children": int(len(man)), "n_clips": int(len(clips)),
          "n_cry": int(len(cry)), "problems": problems, "audit": audit.to_dict("records"),
          "test_children": int((man.split == "test").sum()), "external_site": EXTERNAL_SITE}
U.save_json(report, OUT / "audit_report.json")
print(json.dumps({k: v for k, v in report.items() if k != "audit"}, indent=1))
assert not [p for p in problems if "missing columns" in p or "label must" in p], problems


In [ ]:
R.run('nb00')

## Section `nb01`: NB01 - Face pipeline: decode once -> aligned crops + behaviour features

In [ ]:
%%writefile {R.SEC_DIR}/nb01.py
# # NB01 - Face pipeline: decode once -> aligned crops + behaviour features
# Each protocol clip is decoded **once** at 5 fps (max 40 frames = 8 s).  MediaPipe Face
# Landmarker gives 478 landmarks, 52 blendshapes (action-unit proxies), head pose and iris
# position; the face is eye-aligned to a 160 px crop.  Saved per child as uint8 `.npy`
# shards that NB04-NB10 memory-map - no notebook ever decodes video again.
#
# Runs on **CPU** (all cores, one landmarker per process): costs no GPU quota.
# For the real cohort attach the cohort dataset in addition to NB00's output.
#
# Outputs (`/kaggle/working/face`): `crops/ valid/ hand/ boxes/ qual/ bg/ rows/` + `face_index.csv`
# + `face_qc.csv`.  `bg/` holds face-masked background thumbnails for the shortcut test.
# 

import numpy as np, pandas as pd
from pathlib import Path
import asd_face as AF
U.pip_install("mediapipe")
FPS, MAX_T, CROP, N_BG = 5, 40, 160, 4
OUT = U.WORK / "face"; OUT.mkdir(parents=True, exist_ok=True)
man = pd.read_csv(U.find_path("manifest.csv"))
clips = pd.read_csv(U.find_path("manifest.csv").parent / "clips.csv")
root, is_dev = U.locate_cohort()
print("cohort:", root, "DEV" if is_dev else "REAL", "| children", len(man), "| clips", len(clips))
if PREFLIGHT:
    man = man.head(6)
model_path = AF.get_landmarker_model(U.WORK / "face_landmarker.task")


jobs = []
for cid in man.child_id:
    rows = clips[clips.child_id == cid].sort_values("clip_idx").to_dict("records")
    if rows:
        jobs.append((cid, rows, str(root), str(OUT), FPS, MAX_T, CROP, N_BG))
import multiprocessing as mp
results, failed = [], []
ctx = mp.get_context("spawn")   # fork + OpenCV/MediaPipe threads can deadlock on Linux
with U.Timer("face pipeline"):
    with ctx.Pool(os.cpu_count(), initializer=AF._init_worker, initargs=(model_path, CROP), maxtasksperchild=20) as pool:
        handles = [(j[0], pool.apply_async(AF.process_child, (j,))) for j in jobs]
        for k, (cid, h) in enumerate(handles):
            try:
                results.append(h.get(timeout=1800))
            except Exception as e:
                failed.append((cid, repr(e)[:200]))
            if k % 20 == 0:
                print(f"{k+1}/{len(jobs)} children", flush=True)
print("failed children:", failed or "none")
assert results, "face pipeline produced nothing"
index = pd.DataFrame([row for r in results for row in r["rows"]])
index = index.sort_values(["child_id", "clip_idx"]).reset_index(drop=True)
index.to_csv(OUT / "face_index.csv", index=False)
names = next((r["names"] for r in results if r.get("names")), None) or [f"bs_{i}" for i in range(AF.BLENDSHAPES)] + AF.EXTRA_NAMES
(OUT / "hand_feature_names.json").write_text(json.dumps(names))
print(index.head())


# ## QC: detection rate per clip and per group
# If the face-detection rate differs between ASD and non-ASD children, the detector itself
# is a confound (it is reported, and NB09 tests whether it predicts the label).
# 

index["valid_frac"] = index.n_valid / index.n_frames.clip(lower=1)
qc = index.merge(man[["child_id", "label", "sex", "age_band"]], on="child_id")
print(qc.groupby("label").valid_frac.describe())
qc["repeat_clip"] = qc.valid_frac < 0.5
print("clips to repeat (face visible < 50 %):", int(qc.repeat_clip.sum()))
qc.to_csv(OUT / "face_qc.csv", index=False)
dups = qc[qc.phash != "0"].groupby("phash").child_id.nunique()
print("identical first-crop hashes across children:", int((dups > 1).sum()))
sizes = {d.name: round(sum(f.stat().st_size for f in d.glob("*.npy")) / 1e9, 3) for d in OUT.iterdir() if d.is_dir()}
print("output GB:", sizes)


In [ ]:
R.run('nb01')

## Section `nb02`: NB02 - Voice and cry pipeline for the cohort

In [ ]:
%%writefile {R.SEC_DIR}/nb02.py
# # NB02 - Voice and cry pipeline for the cohort
# * **Voice**: the tablet microphone during each stimulus clip (`audio_path`, or extracted
#   from the clip video with ffmpeg).  Clips where the child did not vocalise
#   (voiced fraction < 5 %) are treated as *missing*, which is exactly the situation the
#   missing-modality fusion is designed for (pre-verbal children).
# * **Cry**: naturally occurring cries recorded in clinic (never provoked), `cry.csv`.
#
# Per item: log-mel `.npy`, eGeMAPS-v02 (openSMILE), wav2vec2-base layer means, AST
# embedding.  `voice_features.npy` / `cry_features.npy` (+ `_index.csv` with `child_id`,
# `clip_idx`) are what the fusion model reads.
# 

import numpy as np, pandas as pd
from pathlib import Path
import asd_audio as A
U.pip_install("opensmile")
OUT = U.WORK / "audio"; OUT.mkdir(parents=True, exist_ok=True)
MIN_VOICED = 0.05
mdir = U.find_path("manifest.csv").parent
man = pd.read_csv(mdir / "manifest.csv")
clips = pd.read_csv(mdir / "clips.csv")
cry = pd.read_csv(mdir / "cry.csv")
root, is_dev = U.locate_cohort()
print("cohort:", root, "DEV" if is_dev else "REAL")
if PREFLIGHT:
    keep = man.child_id.head(8)
    clips, cry = clips[clips.child_id.isin(keep)], cry[cry.child_id.isin(keep)]


clips["audio_src"] = [a if isinstance(a, str) and a.strip() else v for a, v in zip(clips.get("audio_path", pd.Series([""] * len(clips))).fillna(""), clips.video_path)]
with U.Timer("voice"):
    vidx = A.extract_all(clips, "audio_src", root, OUT, "voice", use_ast=not PREFLIGHT, max_s=12)
with U.Timer("cry"):
    cidx = A.extract_all(cry, "cry_path", root, OUT, "cry", use_ast=not PREFLIGHT, max_s=12) if len(cry) else pd.DataFrame()


# ## Fusion-ready vectors (drop non-vocal clips -> missing voice for that clip)
# 

for tag, idx in (("voice", vidx), ("cry", cidx)):
    if not len(idx):
        continue
    X = A.fusion_audio_features(OUT, tag)
    ok = (idx.voiced_frac >= MIN_VOICED).values if tag == "voice" else np.ones(len(idx), bool)
    np.save(OUT / f"{tag}_features.npy", X[ok].astype(np.float32))
    idx[ok].reset_index(drop=True).to_csv(OUT / f"{tag}_fusion_index.csv", index=False)  # rows of *_features.npy
    print(tag, "items kept", int(ok.sum()), "/", len(idx), "| dim", X.shape[1])
    m = idx.merge(man[["child_id", "label"]], on="child_id")
    print(m.groupby("label").voiced_frac.describe()[["mean", "50%"]])
cov = pd.DataFrame({"child_id": man.child_id})
for tag in ("voice", "cry"):
    f = OUT / f"{tag}_fusion_index.csv"
    cov[f"has_{tag}"] = cov.child_id.isin(pd.read_csv(f).child_id) if f.exists() else False
print(cov.merge(man[["child_id", "label"]]).groupby("label")[["has_voice", "has_cry"]].mean())
cov.to_csv(OUT / "audio_coverage.csv", index=False)


In [ ]:
R.run('nb02')

## Section `nb05`: NB05 - Per-frame 256-d embedding cache

In [ ]:
%%writefile {R.SEC_DIR}/nb05.py
# # NB05 - Per-frame 256-d embedding cache
# Every valid frame of every clip is embedded **once** with backbones that never saw the
# cohort labels (NB03 expression nets trained on public data only), so the fusion folds in
# NB08 cannot leak through the embedding.  An ImageNet EfficientNet-B0 (PCA -> 256, fitted
# on CV children only) is cached as a control.
#
# Output: `emb/emb_<backbone>.npy` float16 `[n_clips, 40, 256]` + `_index.csv`.
# 

import numpy as np, pandas as pd, torch
from pathlib import Path
import asd_models as M
OUT = U.WORK / "emb"; OUT.mkdir(parents=True, exist_ok=True)
man = pd.read_csv(U.find_path("manifest.csv"))
store = M.FaceStore(U.find_path("face_index.csv").parent)
ids = [c for c in man.child_id if store.has(c)]
if PREFLIGHT:
    ids = ids[:6]
dev = M.device()
CKPT = U.find_path("efficientnet_b0_expr_backbone.pt")
CKPT = CKPT.parent if CKPT else None
print("children:", len(ids), "| NB03 ckpt dir:", CKPT)


for base in ("efficientnet_b0", "mobilenetv3_l"):
    net = M.load_expr_net(base, CKPT) if CKPT else None
    if net is None:
        print("missing NB03 weights for", base, "- skipped")
        continue
    net = net.to(dev).to(memory_format=torch.channels_last)
    with U.Timer(f"embed {base}_expr"):
        M.embed_store(net, store, ids, 160, OUT / f"emb_{base}_expr.npy", 256, batch=256, fwd=net.embed)


import timm
from sklearn.decomposition import PCA
net = timm.create_model(M.TIMM_ZOO["efficientnet_b0"], pretrained=True, num_classes=0).to(dev).eval()
tmp = OUT / "_imnet_raw.npy"
with U.Timer("embed imagenet effb0"):
    E = M.embed_store(net, store, ids, 160, tmp, net.num_features, batch=256)
idx = pd.read_csv(str(tmp).replace(".npy", "_index.csv"))
cv_ids = set(man.child_id[man.split == "cv"])
rows = np.flatnonzero(idx.child_id.isin(cv_ids).values)
sample = E[rows].reshape(-1, E.shape[-1]).astype(np.float32)
sample = sample[np.abs(sample).sum(1) > 0]
sample = sample[np.random.default_rng(0).choice(len(sample), min(len(sample), 50000), replace=False)]
pca = PCA(256 if sample.shape[0] > 256 else min(sample.shape) - 1, random_state=0).fit(sample)
Z = np.lib.format.open_memmap(OUT / "emb_imagenet_effb0_pca.npy", mode="w+", dtype=np.float16, shape=E.shape[:2] + (pca.n_components_,))
for i in range(len(E)):
    x = E[i].astype(np.float32)
    nz = np.abs(x).sum(1) > 0
    z = np.zeros((x.shape[0], pca.n_components_), np.float32)
    if nz.any():
        z[nz] = pca.transform(x[nz])
    Z[i] = z.astype(np.float16)
Z.flush()
idx.to_csv(OUT / "emb_imagenet_effb0_pca_index.csv", index=False)
del E; tmp.unlink(); Path(str(tmp).replace(".npy", "_index.csv")).unlink()
print("PCA explained variance (256):", round(float(pca.explained_variance_ratio_.sum()), 3))
for f in sorted(OUT.glob("*.npy")):
    print(f.name, np.load(f, mmap_mode="r").shape, round(f.stat().st_size / 1e6, 1), "MB")


In [ ]:
R.run('nb05')

## Section `nb07`: NB07 - Voice and cry baselines for ASD vs non-ASD (child level)

In [ ]:
%%writefile {R.SEC_DIR}/nb07.py
# # NB07 - Voice and cry baselines for ASD vs non-ASD (child level)
# * logistic regression on child-mean eGeMAPS / wav2vec2 / AST vectors
# * EfficientNet-B0 log-mel CNN per clip (one fold per T4), clip probabilities averaged per child
#
# Children with no usable voice (or no cry) are *excluded from that modality's baseline* and
# counted - single-modality models cannot score them, which is the motivation for XMF.
# 

import numpy as np, pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import asd_audio as A
OUT = U.WORK / "nb07"; OUT.mkdir(parents=True, exist_ok=True)
MAN = U.find_path("manifest.csv")
AUD = U.find_path("voice_index.csv").parent
man = pd.read_csv(MAN)
FOLDS = [0] if PREFLIGHT else sorted(man.fold_s0[man.fold_s0 >= 0].unique())
SEEDS = (0,) if PREFLIGHT else (0, 1, 2)


def child_vectors(tag, name):
    p = AUD / f"{tag}_{name}.npy"
    if not p.exists():
        return None
    X = np.load(p).astype(np.float32)
    if name == "w2v":
        X = X[:, 4:10].mean(1)
    idx = pd.read_csv(AUD / f"{tag}_index.csv")
    if tag == "voice":
        ok = idx.voiced_frac.values >= 0.05
        X, idx = X[ok], idx[ok]
    df = pd.DataFrame(np.nan_to_num(X), index=idx.child_id.values)
    return df.groupby(level=0).mean()

coverage = {}
for tag in ("voice", "cry"):
    for name in ("egemaps", "w2v", "ast"):
        V = child_vectors(tag, name)
        if V is None:
            continue
        coverage[tag] = len(V)
        for s in SEEDS:
            for f in FOLDS:
                tr, va, te = U.fold_ids(man, s, f)
                tr, va, te = [[c for c in ids if c in V.index] for ids in (tr, va, te)]
                ytr = man.set_index("child_id").loc[tr, "label"].values
                if len(set(ytr)) < 2:
                    continue
                clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=3000, class_weight="balanced")).fit(V.loc[tr], ytr)
                for part, ids in (("val", va), ("test", te)):
                    if ids:
                        d = pd.DataFrame({"child_id": ids, "prob": clf.predict_proba(V.loc[ids])[:, 1], "part": part})
                        d["y_true"] = d.child_id.map(man.set_index("child_id").label)
                        U.save_preds(d, f"{tag}_logreg_{name}", f, OUT / "preds", s)
print("children with modality:", coverage, "of", len(man))


# ## Log-mel CNN per clip
# 

jobs = []
for tag in ("voice", "cry"):
    ip = AUD / f"{tag}_index.csv"
    if not ip.exists():
        continue
    idx = pd.read_csv(ip).merge(man[["child_id", "label"]], on="child_id")
    if tag == "voice":
        idx = idx[idx.voiced_frac >= 0.05]
    lab_path = OUT / f"{tag}_items_labelled.csv"
    full = pd.read_csv(ip).merge(man[["child_id", "label"]], on="child_id", how="left")
    full.to_csv(lab_path, index=False)
    for f in FOLDS:
        tr, va, te = U.fold_ids(man, 0, f)
        rows = lambda ids: idx.index[idx.child_id.isin(ids)].tolist()
        jobs.append({"name": f"melcnn_{tag}_f{f}", "index": str(lab_path), "mel_dir": str(AUD / f"{tag}_mel"), "label_col": "label",
                     "train_rows": rows(tr), "test_rows": rows(va) + rows(te), "fold": int(f), "out_dir": str(OUT),
                     "seed": 0, "epochs": 1 if PREFLIGHT else 25, "batch": 32, "lr": 1e-3, "max_steps": 3 if PREFLIGHT else None,
                     "crop_frames": 300})
jobs = [j for j in jobs if len(j["train_rows"]) >= 4]
U.run_jobs_parallel("asd_audio", "melcnn_fold_task", jobs, log_dir=OUT / "logs")
for j in jobs:
    p = OUT / "preds" / f"{j['name']}.csv"
    if not p.exists():
        continue
    d = pd.read_csv(p)
    if "p_1" not in d:
        continue
    items = pd.read_csv(j["index"])
    d["child_id"] = items.child_id.values[d.row.values]
    tr, va, te = U.fold_ids(man, 0, j["fold"])
    d["part"] = np.where(d.child_id.isin(va), "val", "test")
    c = d.groupby(["child_id", "part"], as_index=False).agg(prob=("p_1", "mean"), uncertainty=("p_1", "std"))
    c["y_true"] = c.child_id.map(man.set_index("child_id").label)
    U.save_preds(c, f"{j['name'].rsplit('_f', 1)[0]}", j["fold"], OUT / "preds", 0)
    p.unlink()


preds = U.load_all_preds([OUT / "preds"])
rows = []
for m, g in preds.groupby("model"):
    s = U.summarise_model(g, n_boot=200 if PREFLIGHT else 1000)
    rows.append({"model": m, "children": g.child_id.nunique(), "oof_auroc": s.get("oof", {}).get("auroc"), "test_auroc": s.get("test", {}).get("auroc")})
print(U.table_md(pd.DataFrame(rows)))
if man.is_synthetic.iloc[0]:
    print(U.SYNTHETIC_BANNER)


In [ ]:
R.run('nb07')

## Section `nb06`: NB06 - Temporal / video baselines

In [ ]:
%%writefile {R.SEC_DIR}/nb06.py
# # NB06 - Temporal / video baselines
# * **Temporal transformer** on cached NB05 embeddings + behaviour features (face branch of
#   XMF alone) - cheap, 5 folds x 3 seeds.
# * **R(2+1)D-18** (Kinetics), **X3D-S** (PyTorchVideo hub), **VideoMAE-S** (HF) fine-tuned
#   end-to-end on 16-frame face clips - expensive, seed 0 only.
# Same locked child-wise folds as every other notebook.
# 

import numpy as np, pandas as pd
from pathlib import Path
import asd_models as M
OUT = U.WORK / "nb06"; OUT.mkdir(parents=True, exist_ok=True)
MAN = U.find_path("manifest.csv")
FACE = U.find_path("face_index.csv").parent
EMB = U.find_path("emb_efficientnet_b0_expr.npy") or U.find_path("emb_imagenet_effb0_pca.npy")  # fallback if NB03 weights are missing
print("face embedding:", EMB)
man = pd.read_csv(MAN)
FOLDS = [0] if PREFLIGHT else sorted(man.fold_s0[man.fold_s0 >= 0].unique())
SEEDS = (0,) if PREFLIGHT else (0, 1, 2)


# ## 1. Temporal transformer on embeddings
# 

bundle = OUT / "bundle_face.npz"
M.build_fusion_bundle(MAN, EMB, FACE, None, out_path=bundle, quest_cols=[])
jobs = [{"name": f"tt_s{s}_f{f}", "model": "xmf", "model_tag": "video_temporal_transformer", "seed": s, "fold": int(f),
         "bundle": str(bundle), "manifest": str(MAN), "out_dir": str(OUT),
         "cfg": {"mods": ["face"], "no_evidential": True, "modality_dropout": 0.0, "epochs": 3 if PREFLIGHT else 150}}
        for s in SEEDS for f in FOLDS]
U.run_jobs_parallel("asd_models", "fusion_fold_task", jobs, log_dir=OUT / "logs")


# ## 2. End-to-end video CNN / transformer baselines
# 

U.pip_install("pytorchvideo")
VIDEO = {"r2plus1d_18": dict(batch=16, lr=3e-4), "x3d_s": dict(batch=16, lr=3e-4), "videomae_s": dict(batch=8, lr=5e-5)}
jobs = [{"name": f"{m}_s0_f{f}", "model": m, "model_tag": f"video_{m}", "seed": 0, "fold": int(f),
         "manifest": str(MAN), "face_root": str(FACE), "out_dir": str(OUT),
         "epochs": 1 if PREFLIGHT else 10, "patience": 3, "max_steps": 2 if PREFLIGHT else None, **kw}
        for m, kw in VIDEO.items() for f in FOLDS]
if PREFLIGHT:
    jobs = [j for j in jobs if j["model"] == "r2plus1d_18"]
codes = U.run_jobs_parallel("asd_models", "video_fold_task", jobs, log_dir=OUT / "logs")
print("exit codes", codes)


preds = U.load_all_preds([OUT])
rows = []
for m, g in preds.groupby("model"):
    s = U.summarise_model(g, n_boot=200 if PREFLIGHT else 1000)
    rows.append({"model": m, "oof_auroc": s.get("oof", {}).get("auroc"), "test_auroc": s.get("test", {}).get("auroc")})
print(U.table_md(pd.DataFrame(rows)))
if man.is_synthetic.iloc[0]:
    print(U.SYNTHETIC_BANNER)


In [ ]:
R.run('nb06')

## Section `nb04a`: NB04 - Frame-level baselines (one model family per notebook) (a-cnn)

In [ ]:
%%writefile {R.SEC_DIR}/nb04a.py
# # NB04 - Frame-level baselines (one model family per notebook)
# Static per-frame classifiers on the aligned face crops: the comparison models of plan 5.3.
# Every model uses **the same** locked child-wise folds, 160 px input, epoch budget, early
# stopping on validation AUROC (child level) and seeds.  A child's score is the mean over its
# clips of the mean frame probability.
#
# Speed: crops are memory-mapped uint8, augmentation runs on the GPU, fp16 autocast,
# channels_last, linear-probe epoch then full fine-tune, and two folds train at once (one per T4).
# `*_expr` models start from the NB03 expression weights; the rest from ImageNet.
# 

import numpy as np, pandas as pd
from pathlib import Path
if any(m.startswith("yolov8") for m in VARIANT["MODELS"]):
    U.pip_install("ultralytics")
OUT = U.WORK / "nb04"; OUT.mkdir(parents=True, exist_ok=True)
MAN = U.find_path("manifest.csv")
FACE = U.find_path("face_index.csv").parent
man = pd.read_csv(MAN)
SEEDS = (0,) if PREFLIGHT else (0,)      # frame baselines: seed 0; set (0, 1, 2) if quota allows
FOLDS = [0] if PREFLIGHT else sorted(man.fold_s0[man.fold_s0 >= 0].unique())
print("models:", VARIANT["MODELS"], "| folds:", FOLDS, "| synthetic:", bool(man.is_synthetic.iloc[0]))

def init_for(model):
    if not model.endswith("_expr"):
        return None
    p = U.find_path(f"{model[:-5]}_expr_backbone.pt")
    print(model, "init:", p)
    return str(p) if p else None

jobs = []
for model in VARIANT["MODELS"]:
    for s in SEEDS:
        for f in FOLDS:
            jobs.append({"name": f"{model}_s{s}_f{f}", "model": model, "model_tag": f"frame_{model}", "seed": s, "fold": int(f),
                         "manifest": str(MAN), "face_root": str(FACE), "out_dir": str(OUT), "init_weights": init_for(model),
                         "img_size": 160, "epochs": 1 if PREFLIGHT else 12, "patience": 4, "frozen_epochs": 1,
                         "batch": 16 if PREFLIGHT else (96 if model in ("vit_s16", "deit_s", "swin_t", "convnext_tiny") else 128),
                         "lr": 1e-3, "backbone_mult": 0.1, "k_per_clip": 8, "workers": 2,
                         "max_steps": 4 if PREFLIGHT else None,
                         "keep_ckpt": (model == "efficientnet_b0_expr" and s == 0)})   # NB10 Grad-CAM uses these
with U.Timer("frame baselines"):
    codes = U.run_jobs_parallel("asd_models", "frame_fold_task", jobs, log_dir=OUT / "logs")
print("exit codes:", codes)


preds = U.load_all_preds([OUT])
rows = []
for m, g in preds.groupby("model"):
    s = U.summarise_model(g, n_boot=200 if PREFLIGHT else 1000)
    rows.append({"model": m, "oof_auroc": s.get("oof", {}).get("auroc"), "test_auroc": s.get("test", {}).get("auroc"),
                 "test_sens": s.get("test", {}).get("sens"), "test_spec": s.get("test", {}).get("spec")})
res = pd.DataFrame(rows)
print(U.table_md(res))
res.to_csv(OUT / "frame_baselines_summary.csv", index=False)
if man.is_synthetic.iloc[0]:
    print(U.SYNTHETIC_BANNER)


In [ ]:
R.run('nb04a', {"MODELS": ["resnet18", "resnet50", "efficientnet_b0", "efficientnet_b0_expr"]})

## Section `nb04b`: NB04 - Frame-level baselines (one model family per notebook) (b-cnn)

In [ ]:
%%writefile {R.SEC_DIR}/nb04b.py
# # NB04 - Frame-level baselines (one model family per notebook)
# Static per-frame classifiers on the aligned face crops: the comparison models of plan 5.3.
# Every model uses **the same** locked child-wise folds, 160 px input, epoch budget, early
# stopping on validation AUROC (child level) and seeds.  A child's score is the mean over its
# clips of the mean frame probability.
#
# Speed: crops are memory-mapped uint8, augmentation runs on the GPU, fp16 autocast,
# channels_last, linear-probe epoch then full fine-tune, and two folds train at once (one per T4).
# `*_expr` models start from the NB03 expression weights; the rest from ImageNet.
# 

import numpy as np, pandas as pd
from pathlib import Path
if any(m.startswith("yolov8") for m in VARIANT["MODELS"]):
    U.pip_install("ultralytics")
OUT = U.WORK / "nb04"; OUT.mkdir(parents=True, exist_ok=True)
MAN = U.find_path("manifest.csv")
FACE = U.find_path("face_index.csv").parent
man = pd.read_csv(MAN)
SEEDS = (0,) if PREFLIGHT else (0,)      # frame baselines: seed 0; set (0, 1, 2) if quota allows
FOLDS = [0] if PREFLIGHT else sorted(man.fold_s0[man.fold_s0 >= 0].unique())
print("models:", VARIANT["MODELS"], "| folds:", FOLDS, "| synthetic:", bool(man.is_synthetic.iloc[0]))

def init_for(model):
    if not model.endswith("_expr"):
        return None
    p = U.find_path(f"{model[:-5]}_expr_backbone.pt")
    print(model, "init:", p)
    return str(p) if p else None

jobs = []
for model in VARIANT["MODELS"]:
    for s in SEEDS:
        for f in FOLDS:
            jobs.append({"name": f"{model}_s{s}_f{f}", "model": model, "model_tag": f"frame_{model}", "seed": s, "fold": int(f),
                         "manifest": str(MAN), "face_root": str(FACE), "out_dir": str(OUT), "init_weights": init_for(model),
                         "img_size": 160, "epochs": 1 if PREFLIGHT else 12, "patience": 4, "frozen_epochs": 1,
                         "batch": 16 if PREFLIGHT else (96 if model in ("vit_s16", "deit_s", "swin_t", "convnext_tiny") else 128),
                         "lr": 1e-3, "backbone_mult": 0.1, "k_per_clip": 8, "workers": 2,
                         "max_steps": 4 if PREFLIGHT else None,
                         "keep_ckpt": (model == "efficientnet_b0_expr" and s == 0)})   # NB10 Grad-CAM uses these
with U.Timer("frame baselines"):
    codes = U.run_jobs_parallel("asd_models", "frame_fold_task", jobs, log_dir=OUT / "logs")
print("exit codes:", codes)


preds = U.load_all_preds([OUT])
rows = []
for m, g in preds.groupby("model"):
    s = U.summarise_model(g, n_boot=200 if PREFLIGHT else 1000)
    rows.append({"model": m, "oof_auroc": s.get("oof", {}).get("auroc"), "test_auroc": s.get("test", {}).get("auroc"),
                 "test_sens": s.get("test", {}).get("sens"), "test_spec": s.get("test", {}).get("spec")})
res = pd.DataFrame(rows)
print(U.table_md(res))
res.to_csv(OUT / "frame_baselines_summary.csv", index=False)
if man.is_synthetic.iloc[0]:
    print(U.SYNTHETIC_BANNER)


In [ ]:
R.run('nb04b', {"MODELS": ["efficientnetv2_s", "mobilenetv3_l", "mobilenetv3_l_expr", "convnext_tiny"]})

## Section `nb04c`: NB04 - Frame-level baselines (one model family per notebook) (c-vit)

In [ ]:
%%writefile {R.SEC_DIR}/nb04c.py
# # NB04 - Frame-level baselines (one model family per notebook)
# Static per-frame classifiers on the aligned face crops: the comparison models of plan 5.3.
# Every model uses **the same** locked child-wise folds, 160 px input, epoch budget, early
# stopping on validation AUROC (child level) and seeds.  A child's score is the mean over its
# clips of the mean frame probability.
#
# Speed: crops are memory-mapped uint8, augmentation runs on the GPU, fp16 autocast,
# channels_last, linear-probe epoch then full fine-tune, and two folds train at once (one per T4).
# `*_expr` models start from the NB03 expression weights; the rest from ImageNet.
# 

import numpy as np, pandas as pd
from pathlib import Path
if any(m.startswith("yolov8") for m in VARIANT["MODELS"]):
    U.pip_install("ultralytics")
OUT = U.WORK / "nb04"; OUT.mkdir(parents=True, exist_ok=True)
MAN = U.find_path("manifest.csv")
FACE = U.find_path("face_index.csv").parent
man = pd.read_csv(MAN)
SEEDS = (0,) if PREFLIGHT else (0,)      # frame baselines: seed 0; set (0, 1, 2) if quota allows
FOLDS = [0] if PREFLIGHT else sorted(man.fold_s0[man.fold_s0 >= 0].unique())
print("models:", VARIANT["MODELS"], "| folds:", FOLDS, "| synthetic:", bool(man.is_synthetic.iloc[0]))

def init_for(model):
    if not model.endswith("_expr"):
        return None
    p = U.find_path(f"{model[:-5]}_expr_backbone.pt")
    print(model, "init:", p)
    return str(p) if p else None

jobs = []
for model in VARIANT["MODELS"]:
    for s in SEEDS:
        for f in FOLDS:
            jobs.append({"name": f"{model}_s{s}_f{f}", "model": model, "model_tag": f"frame_{model}", "seed": s, "fold": int(f),
                         "manifest": str(MAN), "face_root": str(FACE), "out_dir": str(OUT), "init_weights": init_for(model),
                         "img_size": 160, "epochs": 1 if PREFLIGHT else 12, "patience": 4, "frozen_epochs": 1,
                         "batch": 16 if PREFLIGHT else (96 if model in ("vit_s16", "deit_s", "swin_t", "convnext_tiny") else 128),
                         "lr": 1e-3, "backbone_mult": 0.1, "k_per_clip": 8, "workers": 2,
                         "max_steps": 4 if PREFLIGHT else None,
                         "keep_ckpt": (model == "efficientnet_b0_expr" and s == 0)})   # NB10 Grad-CAM uses these
with U.Timer("frame baselines"):
    codes = U.run_jobs_parallel("asd_models", "frame_fold_task", jobs, log_dir=OUT / "logs")
print("exit codes:", codes)


preds = U.load_all_preds([OUT])
rows = []
for m, g in preds.groupby("model"):
    s = U.summarise_model(g, n_boot=200 if PREFLIGHT else 1000)
    rows.append({"model": m, "oof_auroc": s.get("oof", {}).get("auroc"), "test_auroc": s.get("test", {}).get("auroc"),
                 "test_sens": s.get("test", {}).get("sens"), "test_spec": s.get("test", {}).get("spec")})
res = pd.DataFrame(rows)
print(U.table_md(res))
res.to_csv(OUT / "frame_baselines_summary.csv", index=False)
if man.is_synthetic.iloc[0]:
    print(U.SYNTHETIC_BANNER)


In [ ]:
R.run('nb04c', {"MODELS": ["vit_s16", "deit_s", "swin_t"]})

## Section `nb04d`: NB04 - Frame-level baselines (one model family per notebook) (d-yolo)

In [ ]:
%%writefile {R.SEC_DIR}/nb04d.py
# # NB04 - Frame-level baselines (one model family per notebook)
# Static per-frame classifiers on the aligned face crops: the comparison models of plan 5.3.
# Every model uses **the same** locked child-wise folds, 160 px input, epoch budget, early
# stopping on validation AUROC (child level) and seeds.  A child's score is the mean over its
# clips of the mean frame probability.
#
# Speed: crops are memory-mapped uint8, augmentation runs on the GPU, fp16 autocast,
# channels_last, linear-probe epoch then full fine-tune, and two folds train at once (one per T4).
# `*_expr` models start from the NB03 expression weights; the rest from ImageNet.
# 

import numpy as np, pandas as pd
from pathlib import Path
if any(m.startswith("yolov8") for m in VARIANT["MODELS"]):
    U.pip_install("ultralytics")
OUT = U.WORK / "nb04"; OUT.mkdir(parents=True, exist_ok=True)
MAN = U.find_path("manifest.csv")
FACE = U.find_path("face_index.csv").parent
man = pd.read_csv(MAN)
SEEDS = (0,) if PREFLIGHT else (0,)      # frame baselines: seed 0; set (0, 1, 2) if quota allows
FOLDS = [0] if PREFLIGHT else sorted(man.fold_s0[man.fold_s0 >= 0].unique())
print("models:", VARIANT["MODELS"], "| folds:", FOLDS, "| synthetic:", bool(man.is_synthetic.iloc[0]))

def init_for(model):
    if not model.endswith("_expr"):
        return None
    p = U.find_path(f"{model[:-5]}_expr_backbone.pt")
    print(model, "init:", p)
    return str(p) if p else None

jobs = []
for model in VARIANT["MODELS"]:
    for s in SEEDS:
        for f in FOLDS:
            jobs.append({"name": f"{model}_s{s}_f{f}", "model": model, "model_tag": f"frame_{model}", "seed": s, "fold": int(f),
                         "manifest": str(MAN), "face_root": str(FACE), "out_dir": str(OUT), "init_weights": init_for(model),
                         "img_size": 160, "epochs": 1 if PREFLIGHT else 12, "patience": 4, "frozen_epochs": 1,
                         "batch": 16 if PREFLIGHT else (96 if model in ("vit_s16", "deit_s", "swin_t", "convnext_tiny") else 128),
                         "lr": 1e-3, "backbone_mult": 0.1, "k_per_clip": 8, "workers": 2,
                         "max_steps": 4 if PREFLIGHT else None,
                         "keep_ckpt": (model == "efficientnet_b0_expr" and s == 0)})   # NB10 Grad-CAM uses these
with U.Timer("frame baselines"):
    codes = U.run_jobs_parallel("asd_models", "frame_fold_task", jobs, log_dir=OUT / "logs")
print("exit codes:", codes)


preds = U.load_all_preds([OUT])
rows = []
for m, g in preds.groupby("model"):
    s = U.summarise_model(g, n_boot=200 if PREFLIGHT else 1000)
    rows.append({"model": m, "oof_auroc": s.get("oof", {}).get("auroc"), "test_auroc": s.get("test", {}).get("auroc"),
                 "test_sens": s.get("test", {}).get("sens"), "test_spec": s.get("test", {}).get("spec")})
res = pd.DataFrame(rows)
print(U.table_md(res))
res.to_csv(OUT / "frame_baselines_summary.csv", index=False)
if man.is_synthetic.iloc[0]:
    print(U.SYNTHETIC_BANNER)


In [ ]:
R.run('nb04d', {"MODELS": ["yolov8_cls", "yolov8_att"]})

In [ ]:
R.finish()